In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

df = pd.read_parquet('/kaggle/input/datasets/zxcdfg/dataset-dt1/train.parquet')
df_valid = pd.read_parquet('/kaggle/input/datasets/zxcdfg/dataset-dt1/valid.parquet')

In [2]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

# --------------------------------------------
# 1️⃣ Spread Features
# --------------------------------------------

for data in [df, df_valid]:
    data["p5_minus_p11"] = data["p5"] - data["p11"]
    data["p1_minus_p7"]  = data["p1"] - data["p7"]
    data["dp0_minus_dp2"] = data["dp0"] - data["dp2"]

BASE_FEATURES = [
    "p5_minus_p11",
    "p1_minus_p7",
    "dp0_minus_dp2"
]

LAGS = [1, 2, 3]
ROLL_WINDOWS = [20, 50, 100]

feature_cols = []

# --------------------------------------------
# 2️⃣ Lag Features (Groupwise)
# --------------------------------------------

for col in BASE_FEATURES:
    
    feature_cols.append(col)
    
    for lag in LAGS:
        lag_col = f"{col}_lag{lag}"
        
        df[lag_col] = df.groupby("seq_ix")[col].shift(lag)
        df_valid[lag_col] = df_valid.groupby("seq_ix")[col].shift(lag)
        
        feature_cols.append(lag_col)

# --------------------------------------------
# 3️⃣ Rolling Sum Features (Groupwise)
# --------------------------------------------

for col in BASE_FEATURES:
    
    for window in ROLL_WINDOWS:
        
        roll_col = f"{col}_rollsum_{window}"
        
        df[roll_col] = (
            df.groupby("seq_ix")[col]
            .transform(lambda x: x.rolling(window, min_periods=1).sum())
        )
        
        df_valid[roll_col] = (
            df_valid.groupby("seq_ix")[col]
            .transform(lambda x: x.rolling(window, min_periods=1).sum())
        )
        
        feature_cols.append(roll_col)

# --------------------------------------------
# 4️⃣ Create dt1 (Groupwise)
# --------------------------------------------

df["dt1"] = df.groupby("seq_ix")["t1"].diff()
df_valid["dt1"] = df_valid.groupby("seq_ix")["t1"].diff()

# --------------------------------------------
# 5️⃣ Prepare Training Data
# --------------------------------------------

X_train = df[feature_cols]
y_train = df["dt1"]

mask_train = X_train.notna().all(axis=1) & y_train.notna()

X_train = X_train.loc[mask_train]
y_train = y_train.loc[mask_train]

train_data = lgb.Dataset(X_train, label=y_train)

# --------------------------------------------
# 6️⃣ Train LightGBM
# --------------------------------------------

params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 50,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l2": 0.5,
    "verbosity": -1,
    "seed": 42
}

model = lgb.train(params, train_data, num_boost_round=400)

# --------------------------------------------
# 7️⃣ Validation Prediction
# --------------------------------------------

X_valid = df_valid[feature_cols]
y_valid = df_valid["dt1"]

mask_valid = X_valid.notna().all(axis=1) & y_valid.notna()

X_valid_clean = X_valid.loc[mask_valid]
y_valid_clean = y_valid.loc[mask_valid]

y_pred_valid = model.predict(X_valid_clean)

# --------------------------------------------
# 8️⃣ Metrics
# --------------------------------------------

print("Validation R2:", r2_score(y_valid_clean, y_pred_valid))
print("Validation Pearson:",
      np.corrcoef(y_valid_clean, y_pred_valid)[0,1])
print("Validation Spearman:",
      pd.Series(y_valid_clean.values)
      .corr(pd.Series(y_pred_valid), method="spearman"))

weighted_corr = weighted_pearson_correlation(
    y_valid_clean.values,
    y_pred_valid
)

print("Validation Weighted Pearson:", weighted_corr)

# --------------------------------------------
# 9️⃣ Plot dt1 (One Sequence)
# --------------------------------------------

sample_seq = df_valid.loc[mask_valid, "seq_ix"].iloc[0]

plot_idx = df_valid.loc[mask_valid]
plot_idx = plot_idx[plot_idx["seq_ix"] == sample_seq].index

plt.figure(figsize=(12,6))
plt.plot(df_valid.loc[plot_idx, "step_in_seq"],
         y_valid_clean.loc[plot_idx],
         label="True dt1")

plt.plot(df_valid.loc[plot_idx, "step_in_seq"],
         pd.Series(y_pred_valid, index=X_valid_clean.index).loc[plot_idx],
         label="Pred dt1")

plt.title(f"dt1 Prediction — Sequence {sample_seq}")
plt.legend()
plt.show()


Validation R2: 0.8272838799004106
Validation Pearson: 0.9100739767742678
Validation Spearman: 0.8620811209759569


NameError: name 'weighted_pearson_correlation' is not defined

In [ ]:
# attach predictions
df_valid["dt1_pred"] = np.nan
df_valid.loc[mask_valid, "dt1_pred"] = y_pred_valid

# cumulative sum per sequence
df_valid["t1_reconstructed"] = (
    df_valid
    .groupby("seq_ix")["dt1_pred"]
    .cumsum()
)

# ensure NaNs become 0 at start
df_valid["t1_reconstructed"] = df_valid["t1_reconstructed"].fillna(0)


In [ ]:
mask_recon = df_valid["t1_reconstructed"].notna()

print("Reconstructed Pearson:",
      np.corrcoef(df_valid.loc[mask_recon, "t1"],
                  df_valid.loc[mask_recon, "t1_reconstructed"])[0,1])


In [ ]:
import matplotlib.pyplot as plt

sample_seq = df_valid.loc[mask_recon, "seq_ix"].iloc[0]

plot_df = df_valid[df_valid["seq_ix"] == sample_seq]

plt.figure(figsize=(12,6))
plt.plot(plot_df["step_in_seq"], plot_df["t1"], label="True t1")
plt.plot(plot_df["step_in_seq"], plot_df["t1_reconstructed"],
         label="Reconstructed (0 anchored)")

plt.title(f"Sequence {sample_seq} — Zero Anchored Reconstruction")
plt.legend()
plt.show()


In [ ]:
df_valid["bias"] = (
    df_valid["t1"] - df_valid["t1_reconstructed"]
)


In [ ]:
print("Bias mean:", df_valid.groupby('seq_ix')["bias"].mean().mean())
print("Bias std:", df_valid.groupby('seq_ix')["bias"].mean().std())


In [ ]:
import matplotlib.pyplot as plt

sample_seq = 4

plot_df = df_valid[df_valid["seq_ix"] == sample_seq]

plt.figure(figsize=(12,6))
plt.plot(plot_df["step_in_seq"], plot_df["bias"])
plt.title(f"Bias — Sequence {sample_seq}")
plt.show()


In [ ]:
import numpy as np
import pandas as pd

results = {}

numeric_cols = df_valid.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    
    if col == "bias":
        continue
    
    group_corrs = []
    
    for seq, group in df_valid.groupby("seq_ix"):
        
        if group["bias"].notna().sum() > 2 and group[col].notna().sum() > 2:
            
            corr = group[col].corr(group["bias"])
            
            if pd.notna(corr):
                group_corrs.append(corr)
    
    if len(group_corrs) > 0:
        results[col] = {
            "mean_corr": np.mean(group_corrs),
            "mean_abs_corr": np.mean(np.abs(group_corrs))
        }

corr_df = pd.DataFrame(results).T

# Sort by absolute mean correlation
corr_df = corr_df.sort_values("mean_abs_corr", ascending=False)

print(corr_df.head(20))


In [ ]:
corr_df = corr_df.sort_values("mean_corr", ascending=False)

print(corr_df.head(20))

In [ ]:
# Compute groupwise mean of reconstructed t1
group_mean = (
    df_valid
    .groupby("seq_ix")["t1_reconstructed"]
    .transform("mean")
)

# Create new demeaned column
df_valid["t1_reconstructed_demeaned"] = (
    df_valid["t1_reconstructed"] - group_mean
)


In [ ]:
import sys
sys.path.append('/kaggle/input/datasets/zxcdfg/dataset-dt1')
from utils import weighted_pearson_correlation

In [ ]:
print(weighted_pearson_correlation(df_valid['t1_reconstructed_demeaned'], df_valid['t1']))

In [ ]:
import matplotlib.pyplot as plt

sample_seq = df_valid["seq_ix"].iloc[0]

plot_df = df_valid[df_valid["seq_ix"] == sample_seq]

plt.figure(figsize=(12,6))
plt.plot(plot_df["step_in_seq"], plot_df["t1_reconstructed_demeaned"],
         label="Reconstructed")
plt.plot(plot_df["step_in_seq"], plot_df["t1"],
         label="Demeaned")

plt.title(f"Sequence {sample_seq}")
plt.legend()
plt.show()


In [ ]:
# 1. Standard Pearson between pred and demeaned
print("Normal Pearson:",
      np.corrcoef(y_pred_valid,
                  y_pred_valid - y_pred_valid.mean())[0,1])

# 2. Weighted Pearson between pred and demeaned
print("Weighted Pearson:",
      weighted_pearson_correlation(
          y_pred_valid,
          y_pred_valid - y_pred_valid.mean()
      ))


In [ ]:
df_valid.groupby("seq_ix")["bias"].mean().abs().mean()


In [ ]:
# Only use rows where we have prediction
mask = df_valid["t1_reconstructed"].notna()

y_true = df_valid.loc[mask, "t1"].values
y_pred = df_valid.loc[mask, "t1_reconstructed"].values

# same weights as metric
weights = np.abs(y_true)
weights = np.maximum(weights, 1e-8)

weighted_mean_pred = np.sum(weights * y_pred) / np.sum(weights)

print("Weighted global bias:", weighted_mean_pred)


In [ ]:
df_valid["t1_pred_global_bias"] = df_valid["t1_reconstructed"] - df_valid["t1_reconstructed"].mean()



In [ ]:
print("Before bias removal:",
      weighted_pearson_correlation(
          df_valid['t1'],
          df_valid['t1_reconstructed']
      ))

print("After bias removal:",
      weighted_pearson_correlation(
          df_valid['t1'],
          df_valid.loc[mask, "t1_pred_global_bias"].values
      ))


In [ ]:
for seq, group in df_valid.groupby("seq_ix"):
    
    group = group.sort_values("step_in_seq")
    
    t100 = group.loc[group["step_in_seq"] == 1000, "t1_reconstructed"]
    
    if len(t100) == 0:
        continue
    
    t100 = t100.values[0]
    
    slope_est = t100 / 1000
    
    correction = slope_est * group["step_in_seq"]
    
    df_valid.loc[group.index, "t1_pred_linear_corrected"] = (
        group["t1_reconstructed"] - correction
    )


In [ ]:
print(weighted_pearson_correlation(df_valid['t1_pred_linear_corrected'], df_valid['t1']))

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

WINDOW = 50
BATCH_SIZE = 512
EPOCHS = 5
LR = 1e-3

# --------------------------------------------
# 1️⃣ Create Spread Features
# --------------------------------------------

for data in [df, df_valid]:
    data["p5_minus_p11"] = data["p5"] - data["p11"]
    data["p1_minus_p7"]  = data["p1"] - data["p7"]
    data["dp0_minus_dp2"] = data["dp0"] - data["dp2"]

FEATURES = [
    "p5_minus_p11",
    "p1_minus_p7",
    "dp0_minus_dp2"
]

# --------------------------------------------
# 2️⃣ Build Sliding Windows
# --------------------------------------------

def build_sequences(data):
    X, y = [], []
    
    for seq, group in data.groupby("seq_ix"):
        group = group.sort_values("step_in_seq")
        
        values = group[FEATURES].values
        targets = group["t1"].values
        
        for i in range(WINDOW, len(group)):
            X.append(values[i-WINDOW:i])
            y.append(targets[i])
    
    return np.array(X), np.array(y)

X_train, y_train = build_sequences(df)
X_valid, y_valid = build_sequences(df_valid)

# --------------------------------------------
# 3️⃣ Dataset
# --------------------------------------------

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(SeqDataset(X_train, y_train),
                          batch_size=BATCH_SIZE,
                          shuffle=True)

valid_loader = DataLoader(SeqDataset(X_valid, y_valid),
                          batch_size=BATCH_SIZE,
                          shuffle=False)

# --------------------------------------------
# 4️⃣ Transformer Model
# --------------------------------------------

class TransformerModel(nn.Module):
    def __init__(self, input_dim=3, d_model=64, nhead=4, num_layers=2):
        super().__init__()
        
        self.input_proj = nn.Linear(input_dim, d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True
        )
        
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )
        
        self.fc = nn.Linear(d_model, 1)
    
    def forward(self, x):
        x = self.input_proj(x)
        x = self.transformer(x)
        x = x[:, -1, :]   # last timestep
        return self.fc(x).squeeze(-1)

model = TransformerModel().to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# --------------------------------------------
# 5️⃣ Training Loop
# --------------------------------------------

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.6f}")

# --------------------------------------------
# 6️⃣ Validation
# --------------------------------------------

model.eval()
preds = []

with torch.no_grad():
    for xb, yb in valid_loader:
        xb = xb.to(device)
        pred = model(xb)
        preds.append(pred.cpu().numpy())

y_pred = np.concatenate(preds)

print("Validation R2:", r2_score(y_valid, y_pred))
print("Validation Pearson:",
      np.corrcoef(y_valid, y_pred)[0,1])


In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import r2_score

# --------------------------------------------------
# 1️⃣ Spread Features
# --------------------------------------------------

for data in [df, df_valid]:
    data["p5_minus_p11"] = data["p5"] - data["p11"]
    data["p1_minus_p7"]  = data["p1"] - data["p7"]
    data["dp0_minus_dp2"] = data["dp0"] - data["dp2"]

BASE_FEATURES = [
    "p5_minus_p11",
    "p1_minus_p7",
    "dp0_minus_dp2"
]

LAGS = [1,2,3]
feature_cols = []

for col in BASE_FEATURES:
    
    feature_cols.append(col)
    
    for lag in LAGS:
        lag_col = f"{col}_lag{lag}"
        
        df[lag_col] = df.groupby("seq_ix")[col].shift(lag)
        df_valid[lag_col] = df_valid.groupby("seq_ix")[col].shift(lag)
        
        feature_cols.append(lag_col)

# --------------------------------------------------
# 2️⃣ Create dt1 (groupwise)
# --------------------------------------------------

df["dt1"] = df.groupby("seq_ix")["t1"].diff()
df_valid["dt1"] = df_valid.groupby("seq_ix")["t1"].diff()

# --------------------------------------------------
# 3️⃣ Remove GLOBAL TRAINING MEAN (NO LEAKAGE)
# --------------------------------------------------

train_dt1_mean = df["dt1"].mean()
print("Training dt1 mean:", train_dt1_mean)

df["dt1_centered"] = df["dt1"] - train_dt1_mean

# IMPORTANT:
# We subtract SAME training mean from validation target
df_valid["dt1_centered"] = df_valid["dt1"] - train_dt1_mean

# --------------------------------------------------
# 4️⃣ Prepare Train Data
# --------------------------------------------------

X_train = df[feature_cols]
y_train = df["dt1_centered"]

mask_train = X_train.notna().all(axis=1) & y_train.notna()

X_train = X_train.loc[mask_train]
y_train = y_train.loc[mask_train]

train_data = lgb.Dataset(X_train, label=y_train)

# --------------------------------------------------
# 5️⃣ Train LightGBM
# --------------------------------------------------

params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 50,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l2": 0.5,
    "verbosity": -1,
    "seed": 42
}

model = lgb.train(params, train_data, num_boost_round=400)

# --------------------------------------------------
# 6️⃣ Validation Prediction
# --------------------------------------------------

X_valid = df_valid[feature_cols]
y_valid = df_valid["dt1_centered"]

mask_valid = X_valid.notna().all(axis=1) & y_valid.notna()

X_valid_clean = X_valid.loc[mask_valid]
y_valid_clean = y_valid.loc[mask_valid]

y_pred_valid = model.predict(X_valid_clean)

# --------------------------------------------------
# 7️⃣ Add Back Mean (FOR PROPER DT1)
# --------------------------------------------------

y_pred_valid = y_pred_valid + train_dt1_mean

# --------------------------------------------------
# 8️⃣ Metrics
# --------------------------------------------------

print("Validation R2:", r2_score(df_valid.loc[mask_valid,"dt1"], y_pred_valid))
print("Validation Pearson:",
      np.corrcoef(df_valid.loc[mask_valid,"dt1"], y_pred_valid)[0,1])


In [ ]:
import numpy as np
import pandas as pd

# --------------------------------------------------
# 1️⃣ Create spreads
# --------------------------------------------------

for data in [df]:
    data["p1_minus_p11"] = data["p1"] - data["p11"]
    data["p5_minus_p11"] = data["p5"] - data["p11"]
    data["p4_minus_p11"] = data["p4"] - data["p11"]
    data["p1_minus_p7"]  = data["p1"] - data["p7"]
    data["p2_minus_p11"] = data["p2"] - data["p11"]

SPREADS = [
    "p1_minus_p11",
    "p5_minus_p11",
    "p4_minus_p11",
    "p1_minus_p7",
    "p2_minus_p11"
]

# --------------------------------------------------
# 2️⃣ Create dt1 (groupwise)
# --------------------------------------------------

df["dt1"] = df.groupby("seq_ix")["t1"].diff()

# --------------------------------------------------
# 3️⃣ Groupwise Correlation with dt1
# --------------------------------------------------

results = {}

for col in SPREADS:
    
    group_corrs = []
    
    for seq, group in df.groupby("seq_ix"):
        
        x = group[col]
        y = group["dt1"]
        
        valid = x.notna() & y.notna()
        x = x[valid]
        y = y[valid]
        
        if len(x) < 3:
            continue
        
        if x.std() == 0 or y.std() == 0:
            continue
        
        corr = np.corrcoef(x, y)[0,1]
        group_corrs.append(corr)
    
    if len(group_corrs) > 0:
        results[col] = {
            "mean_corr": np.mean(group_corrs),
            "mean_abs_corr": np.mean(np.abs(group_corrs))
        }

corr_df = pd.DataFrame(results).T
corr_df = corr_df.sort_values("mean_abs_corr", ascending=False)

print(corr_df)


In [ ]:
print(df["p1_minus_p11"].corr(df["t1"]))


In [ ]:
import numpy as np
import pandas as pd

# --------------------------------------------------
# 1️⃣ Create spreads
# --------------------------------------------------

df["p1_minus_p11"] = df["p1"] - df["p11"]
df["p5_minus_p11"] = df["p5"] - df["p11"]
df["p4_minus_p11"] = df["p4"] - df["p11"]
df["p1_minus_p7"]  = df["p1"] - df["p7"]
df["p2_minus_p11"] = df["p2"] - df["p11"]

SPREADS = [
    "p1_minus_p11",
    "p5_minus_p11",
    "p4_minus_p11",
    "p1_minus_p7",
    "p2_minus_p11"
]

# --------------------------------------------------
# 2️⃣ Create dt1 and FUTURE dt1
# --------------------------------------------------

df["dt1"] = df.groupby("seq_ix")["t1"].diff()

# future change (predictive target)
df["dt1_future"] = df.groupby("seq_ix")["dt1"].shift(-1)

# --------------------------------------------------
# 3️⃣ Groupwise Predictive Correlation
# --------------------------------------------------

results = []

for col in SPREADS:
    
    tmp = df[["seq_ix", col, "dt1_future"]].dropna()
    
    grouped = tmp.groupby("seq_ix")
    
    # compute correlation inside each sequence
    corr = grouped.apply(lambda g: g[col].corr(g["dt1_future"]))
    
    results.append({
        "feature": col,
        "mean_corr": corr.mean(),
        "mean_abs_corr": corr.abs().mean()
    })

corr_df = pd.DataFrame(results).sort_values("mean_abs_corr", ascending=False)

print(corr_df)


In [ ]:
print(df["dt1"].std())
print(df["p1_minus_p11"].std())


In [ ]:
print("Mean dt1:", df["dt1"].mean())
print("Std dt1:", df["dt1"].std())


In [ ]:
print("Mean dt1_pred:", dt1_pred.mean())


In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb

# --------------------------------------------------
# 1️⃣ Features
# --------------------------------------------------

for data in [df, df_valid]:
    data["p1_minus_p11"] = data["p1"] - data["p11"]
    data["p5_minus_p11"] = data["p5"] - data["p11"]
    data["p4_minus_p11"] = data["p4"] - data["p11"]
    data["p1_minus_p7"]  = data["p1"] - data["p7"]
    data["p2_minus_p11"] = data["p2"] - data["p11"]
    data["dp0_minus_dp2"] = data["dp0"] - data["dp2"]

FEATURES = [
    "p1_minus_p11",
    "p5_minus_p11",
    "p4_minus_p11",
    "p1_minus_p7",
    "p2_minus_p11",
    "dp0_minus_dp2"
]

# --------------------------------------------------
# 2️⃣ Target
# --------------------------------------------------

df["dt1"] = df.groupby("seq_ix")["t1"].diff()
df_valid["dt1"] = df_valid.groupby("seq_ix")["t1"].diff()

# Center TRAIN target only
train_mean = df["dt1"].mean()
df["dt1_centered"] = df["dt1"]

# --------------------------------------------------
# 3️⃣ Prepare Data
# --------------------------------------------------

X_train = df[FEATURES]
y_train = df["dt1_centered"]

mask_train = X_train.notna().all(axis=1) & y_train.notna()
X_train = X_train.loc[mask_train]
y_train = y_train.loc[mask_train]

train_data = lgb.Dataset(X_train, label=y_train)

# --------------------------------------------------
# 4️⃣ Train WITHOUT intercept
# --------------------------------------------------

params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 50,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l2": 0.5,
    "verbosity": -1,
    "seed": 42,
    "boost_from_average": False  # CRITICAL
}

model = lgb.train(params, train_data, num_boost_round=400)

# --------------------------------------------------
# 5️⃣ Predict
# --------------------------------------------------

X_valid = df_valid[FEATURES]
mask_valid = X_valid.notna().all(axis=1)

dt1_pred = model.predict(X_valid.loc[mask_valid])

# add back training mean (legal)
dt1_pred = dt1_pred + train_mean

df_valid.loc[mask_valid, "dt1_pred"] = dt1_pred

# --------------------------------------------------
# 6️⃣ Reconstruct t1
# --------------------------------------------------

df_valid["t1_reconstructed"] = (
    df_valid.groupby("seq_ix")["dt1_pred"].cumsum()
)


In [ ]:
df_valid

In [ ]:
print(weighted_pearson_correlation(df_valid['t1'], df_valid['t1_reconstructed']))

In [ ]:
df.groupby('seq_ix')['dt1'].mean().std()

In [ ]:
import matplotlib.pyplot as plt

# pick any sequence you want
sample_seq = df_valid["seq_ix"].iloc[0]   # change index if needed

plot_df = df_valid[df_valid["seq_ix"] == sample_seq].copy()
plot_df = plot_df.sort_values("step_in_seq")

plt.figure(figsize=(12,6))

plt.plot(plot_df["step_in_seq"],
         plot_df["t1"],
         label="True t1",
         linewidth=2)

plt.plot(plot_df["step_in_seq"],
         plot_df["t1_reconstructed"],
         label="Reconstructed t1",
         linewidth=2)

plt.legend()
plt.title(f"Sequence {sample_seq}")
plt.xlabel("Step in Sequence")
plt.ylabel("t1")
plt.grid(True)
plt.show()


In [ ]:
import numpy as np777
    "p5_minus_p11",
    "p4_minus_p11",
    "p1_minus_p7",
    "p2_minus_p11",
    "dp0_minus_dp2"
]

# -----------------------------
# 2. Target
# -----------------------------

df["dt1"] = df.groupby("seq_ix")["t1"].diff()
df_valid["dt1"] = df_valid.groupby("seq_ix")["t1"].diff()

# -----------------------------
# 3. Prepare Data
# -----------------------------

X_train = df[FEATURES]
y_train = df["dt1"]

mask_train = X_train.notna().all(axis=1) & y_train.notna()
X_train = X_train.loc[mask_train]
y_train = y_train.loc[mask_train]

# -----------------------------
# 4. Train Ridge (NO INTERCEPT)
# -----------------------------

model = Ridge(alpha=1.0, fit_intercept=False)
model.fit(X_train, y_train)

# -----------------------------
# 5. Predict
# -----------------------------

X_valid = df_valid[FEATURES]
mask_valid = X_valid.notna().all(axis=1)

dt1_pred = model.predict(X_valid.loc[mask_valid])

df_valid.loc[mask_valid, "dt1_pred"] = dt1_pred

# -----------------------------
# 6. Reconstruct t1
# -----------------------------

df_valid["t1_reconstructed"] = (
    df_valid.groupby("seq_ix")["dt1_pred"].cumsum()
)

# -----------------------------
# 7. Metrics
# -----------------------------

print("dt1 Pearson:",
      np.corrcoef(df_valid.loc[mask_valid, "dt1"], dt1_pred)[0,1])

print("dt1 R2:",
      r2_score(df_valid.loc[mask_valid, "dt1"], dt1_pred))


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

# --------------------------------------------------
# 1️⃣ Create Features
# --------------------------------------------------

for data in [df, df_valid]:
    data["p1_minus_p11"] = data["p1"] - data["p11"]
    data["p5_minus_p11"] = data["p5"] - data["p11"]
    data["p4_minus_p11"] = data["p4"] - data["p11"]
    data["p1_minus_p7"]  = data["p1"] - data["p7"]
    data["p2_minus_p11"] = data["p2"] - data["p11"]
    data["dp0_minus_dp2"] = data["dp0"] - data["dp2"]

FEATURES = [
    "p1_minus_p11",
    "p5_minus_p11",
    "p4_minus_p11",
    "p1_minus_p7",
    "p2_minus_p11",
    "dp0_minus_dp2"
]

# --------------------------------------------------
# 2️⃣ Create dt1
# --------------------------------------------------

df["dt1"] = df.groupby("seq_ix")["t1"].diff()
df_valid["dt1"] = df_valid.groupby("seq_ix")["t1"].diff()

# --------------------------------------------------
# 3️⃣ Convert to numeric (force remove weird objects)
# --------------------------------------------------

for col in FEATURES + ["dt1"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df_valid[col] = pd.to_numeric(df_valid[col], errors="coerce")

# --------------------------------------------------
# 4️⃣ Drop ALL rows with NaNs in required columns
# --------------------------------------------------

train_cols = ["seq_ix"] + FEATURES + ["dt1"]
valid_cols = ["seq_ix"] + FEATURES + ["dt1"]

df_train_clean = df[train_cols].dropna()
df_valid_clean = df_valid[valid_cols].dropna()

# --------------------------------------------------
# 5️⃣ Final NaN verification
# --------------------------------------------------

print("Train NaNs total:",
      df_train_clean.isna().sum().sum())

print("Valid NaNs total:",
      df_valid_clean.isna().sum().sum())

# --------------------------------------------------
# 6️⃣ Prepare matrices
# --------------------------------------------------

X_train = df_train_clean[FEATURES].values
y_train = df_train_clean["dt1"].values

X_valid = df_valid_clean[FEATURES].values
y_valid = df_valid_clean["dt1"].values

# Double check
print("X_train NaNs:", np.isnan(X_train).sum())
print("y_train NaNs:", np.isnan(y_train).sum())

# --------------------------------------------------
# 7️⃣ Train Ridge (NO INTERCEPT)
# --------------------------------------------------

model = Ridge(alpha=1.0, fit_intercept=False, solver="sag")
model.fit(X_train, y_train)

# --------------------------------------------------
# 8️⃣ Predict
# --------------------------------------------------

dt1_pred = model.predict(X_valid)

df_valid_clean["dt1_pred"] = dt1_pred

# --------------------------------------------------
# 9️⃣ Reconstruct t1
# --------------------------------------------------

df_valid_clean["t1_reconstructed"] = (
    df_valid_clean.groupby("seq_ix")["dt1_pred"].cumsum()
)

# --------------------------------------------------
# 🔟 Metrics
# --------------------------------------------------

print("dt1 Pearson:",
      np.corrcoef(y_valid, dt1_pred)[0,1])

print("dt1 R2:",
      r2_score(y_valid, dt1_pred))


In [ ]:
df_valid_clean["t1"] = (
    df_valid_clean.groupby("seq_ix")["dt1"].cumsum()
)

In [ ]:
weighted_pearson_correlation(df_valid_clean['t1_reconstructed'], df_valid_clean['t1'])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# pick a sequence
sample_seq = np.random.choice(df_valid_clean["seq_ix"].unique())

plot_df = df_valid_clean[df_valid_clean["seq_ix"] == sample_seq]
plot_df = plot_df.sort_values("step_in_seq")

plt.figure(figsize=(12,6))

plt.plot(plot_df["step_in_seq"],
         plot_df["t1"],
         label="True t1",
         linewidth=2)

plt.plot(plot_df["step_in_seq"],
         plot_df["t1_reconstructed"],
         label="Reconstructed t1",
         linewidth=2)

plt.title(f"Sequence {sample_seq}")
plt.xlabel("Step in Sequence")
plt.ylabel("t1")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Keep EVERYTHING needed for plotting

valid_cols = ["seq_ix", "step_in_seq", "t1"] + FEATURES + ["dt1"]

df_valid_clean = df_valid[valid_cols].dropna()

# Predict again
X_valid = df_valid_clean[FEATURES].values
dt1_pred = model.predict(X_valid)

df_valid_clean["dt1_pred"] = dt1_pred

# Reconstruct t1
df_valid_clean["t1_reconstructed"] = (
    df_valid_clean.groupby("seq_ix")["dt1_pred"].cumsum()
)

# Plot
import matplotlib.pyplot as plt
import numpy as np

sample_seq = 20

plot_df = df_valid_clean[df_valid_clean["seq_ix"] == sample_seq]
plot_df = plot_df.sort_values("step_in_seq")

plt.figure(figsize=(12,6))
plt.plot(plot_df["step_in_seq"], plot_df["t1"], label="True t1", linewidth=2)
plt.plot(plot_df["step_in_seq"], plot_df["t1_reconstructed"], label="Reconstructed t1", linewidth=2)
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import r2_score

# --------------------------------------------------
# 1️⃣ Create Spread Features
# --------------------------------------------------

for data in [df, df_valid]:
    data["p1_minus_p11"] = data["p1"] - data["p11"]
    data["p5_minus_p11"] = data["p5"] - data["p11"]
    data["p4_minus_p11"] = data["p4"] - data["p11"]
    data["p1_minus_p7"]  = data["p1"] - data["p7"]
    data["p2_minus_p11"] = data["p2"] - data["p11"]
    data["dp0_minus_dp2"] = data["dp0"] - data["dp2"]

FEATURES = [
    "p1_minus_p11",
    "p5_minus_p11",
    "p4_minus_p11",
    "p1_minus_p7",
    "p2_minus_p11",
    "dp0_minus_dp2"
]

# --------------------------------------------------
# 2️⃣ Create dt1 (groupwise)
# --------------------------------------------------

df["dt1"] = df.groupby("seq_ix")["t1"].diff()
df_valid["dt1"] = df_valid.groupby("seq_ix")["t1"].diff()

# --------------------------------------------------
# 3️⃣ Clean Data (keep required cols)
# --------------------------------------------------

train_cols = ["seq_ix"] + FEATURES + ["dt1"]
valid_cols = ["seq_ix", "step_in_seq", "t1"] + FEATURES + ["dt1"]

df_train_clean = df[train_cols].dropna()
df_valid_clean = df_valid[valid_cols].dropna()

X_train = df_train_clean[FEATURES].values
y_train = df_train_clean["dt1"].values

X_valid = df_valid_clean[FEATURES].values
y_valid = df_valid_clean["dt1"].values

# --------------------------------------------------
# 4️⃣ Train XGBoost (GPU)
# --------------------------------------------------

model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    tree_method="hist",
    device="cuda",                 # GPU
    objective="reg:squarederror",
    random_state=42
)

model.fit(X_train, y_train)

# --------------------------------------------------
# 5️⃣ Predict dt1
# --------------------------------------------------

df_valid_clean["dt1_pred"] = model.predict(X_valid)

# --------------------------------------------------
# 6️⃣ Reconstruct t1
# --------------------------------------------------

df_valid_clean["t1_reconstructed"] = (
    df_valid_clean.groupby("seq_ix")["dt1_pred"].cumsum()
)

# --------------------------------------------------
# 7️⃣ Metrics
# --------------------------------------------------

print("dt1 Pearson:",
      np.corrcoef(y_valid, df_valid_clean["dt1_pred"])[0,1])

print("dt1 R2:",
      r2_score(y_valid, df_valid_clean["dt1_pred"]))


In [ ]:
df_valid_clean["t1"] = (
    df_valid_clean.groupby("seq_ix")["dt1"].cumsum()
)

In [ ]:
weighted_pearson_correlation(df_valid_clean['t1_reconstructed'], df_valid_clean['t1'])

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------------
# 1️⃣ Features (already created earlier)
# --------------------------------------------------

FEATURES = [
    "p1_minus_p11",
    "p5_minus_p11",
    "p4_minus_p11",
    "p1_minus_p7",
    "p2_minus_p11",
    "dp0_minus_dp2"
]

# --------------------------------------------------
# 2️⃣ Clean Data
# --------------------------------------------------

train_cols = ["seq_ix"] + FEATURES + ["dt1"]
valid_cols = ["seq_ix", "step_in_seq", "t1"] + FEATURES + ["dt1"]

df_train_clean = df[train_cols].dropna()
df_valid_clean = df_valid[valid_cols].dropna()

X_train = torch.tensor(df_train_clean[FEATURES].values, dtype=torch.float32)
y_train = torch.tensor(df_train_clean["dt1"].values, dtype=torch.float32).view(-1,1)

X_valid = torch.tensor(df_valid_clean[FEATURES].values, dtype=torch.float32)
y_valid = torch.tensor(df_valid_clean["dt1"].values, dtype=torch.float32).view(-1,1)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=4096,
    shuffle=True
)

# --------------------------------------------------
# 3️⃣ Define MLP
# --------------------------------------------------

class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        return self.net(x)

model = MLP(len(FEATURES)).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# --------------------------------------------------
# 4️⃣ Train
# --------------------------------------------------

EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.6f}")

# --------------------------------------------------
# 5️⃣ Predict
# --------------------------------------------------

model.eval()
with torch.no_grad():
    dt1_pred = model(X_valid.to(device)).cpu().numpy().flatten()

df_valid_clean["dt1_pred"] = dt1_pred

# --------------------------------------------------
# 6️⃣ Reconstruct t1
# --------------------------------------------------

df_valid_clean["t1_reconstructed"] = (
    df_valid_clean.groupby("seq_ix")["dt1_pred"].cumsum()
)

# --------------------------------------------------
# 7️⃣ Metric
# --------------------------------------------------

print("dt1 Pearson:",
      np.corrcoef(y_valid.numpy().flatten(), dt1_pred)[0,1])


In [ ]:
df_valid_clean["t1"] = (
    df_valid_clean.groupby("seq_ix")["dt1"].cumsum()
)

In [ ]:
weighted_pearson_correlation(df_valid_clean['t1_reconstructed'], df_valid_clean['t1'])

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------------
# 1️⃣ Create Features
# --------------------------------------------------

for data in [df, df_valid]:
    data["p1_minus_p11"] = data["p1"] - data["p11"]
    data["p5_minus_p11"] = data["p5"] - data["p11"]
    data["p4_minus_p11"] = data["p4"] - data["p11"]
    data["p1_minus_p7"]  = data["p1"] - data["p7"]
    data["p2_minus_p11"] = data["p2"] - data["p11"]
    data["dp0_minus_dp2"] = data["dp0"] - data["dp2"]

FEATURES = [
    "p1_minus_p11",
    "p5_minus_p11",
    "p4_minus_p11",
    "p1_minus_p7",
    "p2_minus_p11",
    "dp0_minus_dp2"
]

# --------------------------------------------------
# 2️⃣ Create dt1
# --------------------------------------------------

df["dt1"] = df.groupby("seq_ix")["t1"].diff()
df_valid["dt1"] = df_valid.groupby("seq_ix")["t1"].diff()

# --------------------------------------------------
# 3️⃣ Clean Data
# --------------------------------------------------

train_cols = ["seq_ix"] + FEATURES + ["dt1"]
valid_cols = ["seq_ix", "step_in_seq", "t1"] + FEATURES + ["dt1"]

df_train_clean = df[train_cols].dropna()
df_valid_clean = df_valid[valid_cols].dropna()

# --------------------------------------------------
# 4️⃣ Scale Features
# --------------------------------------------------

scaler = StandardScaler()

X_train_np = scaler.fit_transform(df_train_clean[FEATURES])
X_valid_np = scaler.transform(df_valid_clean[FEATURES])

y_train_np = df_train_clean["dt1"].values
y_valid_np = df_valid_clean["dt1"].values

# Convert to tensors
X_train = torch.tensor(X_train_np, dtype=torch.float32)
y_train = torch.tensor(y_train_np, dtype=torch.float32).view(-1,1)

X_valid = torch.tensor(X_valid_np, dtype=torch.float32)
y_valid = torch.tensor(y_valid_np, dtype=torch.float32).view(-1,1)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=4096,
    shuffle=True
)

# --------------------------------------------------
# 5️⃣ Define Improved MLP
# --------------------------------------------------

class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

model = MLP(len(FEATURES)).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

criterion = nn.SmoothL1Loss()   # Huber loss

# --------------------------------------------------
# 6️⃣ Train
# --------------------------------------------------

EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.6f}")

# --------------------------------------------------
# 7️⃣ Predict
# --------------------------------------------------

model.eval()
with torch.no_grad():
    dt1_pred = model(X_valid.to(device)).cpu().numpy().flatten()

df_valid_clean["dt1_pred"] = dt1_pred

# --------------------------------------------------
# 8️⃣ Reconstruct t1
# --------------------------------------------------

df_valid_clean["t1_reconstructed"] = (
    df_valid_clean.groupby("seq_ix")["dt1_pred"].cumsum()
)

# --------------------------------------------------
# 9️⃣ Metric
# --------------------------------------------------

pearson = np.corrcoef(y_valid_np, dt1_pred)[0,1]
print("dt1 Pearson:", pearson)


In [ ]:
df_valid_clean["t1"] = (
    df_valid_clean.groupby("seq_ix")["dt1"].cumsum()
)

In [ ]:
weighted_pearson_correlation(df_valid_clean['t1_reconstructed'], df_valid_clean['t1'])

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# --------------------------------------------------
# 1️⃣ Define Original Features
# --------------------------------------------------

# assuming original columns are:
# p0–p11, dp0–dp3, v0–v11, dv0–dv3

FEATURES = (
    [f"p{i}" for i in range(12)] +
    [f"dp{i}" for i in range(4)] +
    [f"v{i}" for i in range(12)] +
    [f"dv{i}" for i in range(4)]
)

# --------------------------------------------------
# 2️⃣ Create dt1
# --------------------------------------------------

df["dt1"] = df.groupby("seq_ix")["t1"].diff()
df_valid["dt1"] = df_valid.groupby("seq_ix")["t1"].diff()

# --------------------------------------------------
# 3️⃣ Clean Data
# --------------------------------------------------

train_cols = ["seq_ix"] + FEATURES + ["dt1"]
valid_cols = ["seq_ix", "step_in_seq", "t1"] + FEATURES + ["dt1"]

df_train_clean = df[train_cols].dropna()
df_valid_clean = df_valid[valid_cols].dropna()

X_train = df_train_clean[FEATURES].values
y_train = df_train_clean["dt1"].values

X_valid = df_valid_clean[FEATURES].values
y_valid = df_valid_clean["dt1"].values

# --------------------------------------------------
# 4️⃣ Train Linear Regression
# --------------------------------------------------

model = LinearRegression(fit_intercept=True)
model.fit(X_train, y_train)

# --------------------------------------------------
# 5️⃣ Predict
# --------------------------------------------------

dt1_pred = model.predict(X_valid)

df_valid_clean["dt1_pred"] = dt1_pred

# --------------------------------------------------
# 6️⃣ Reconstruct t1
# --------------------------------------------------

df_valid_clean["t1_reconstructed"] = (
    df_valid_clean.groupby("seq_ix")["dt1_pred"].cumsum()
)

# --------------------------------------------------
# 7️⃣ Metrics
# --------------------------------------------------

print("dt1 Pearson:",
      np.corrcoef(y_valid, dt1_pred)[0,1])

print("dt1 R2:",
      r2_score(y_valid, dt1_pred))
